In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
import uuid
 
CONFIG = {
    "catalog": "workspace",
    "schema":  "banking_datasentry",
}
 
REPORTED_AT = datetime.now().isoformat()
 
# Severity rules — based on which dimension the failed rule belongs to
# HIGH   : Referential Integrity, Uniqueness — structural problems
# MEDIUM : Validity, Consistency — format and logic problems
# LOW    : Completeness — missing data
 
SEVERITY_MAP = {
    "Referential Integrity": "HIGH",
    "Uniqueness":            "HIGH",
    "Validity":              "MEDIUM",
    "Consistency":           "MEDIUM",
    "Completeness":          "LOW",
}
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Catalog     : {CONFIG['catalog']}")
print(f"Schema      : {CONFIG['schema']}")
print(f"Reported at : {REPORTED_AT}")
print(f"\nSeverity mapping:")
for dim, sev in SEVERITY_MAP.items():
    print(f"  {dim:<25} -> {sev}")

In [0]:
#Rule-to-Dimension map 
RULE_DIMENSION_MAP = {
    # Customers
    "customer_id_not_null":                 "Completeness",
    "full_name_not_null":                   "Completeness",
    "email_not_null":                       "Completeness",
    "created_at_not_null":                  "Completeness",
    "email_format_valid":                   "Validity",
    "date_of_birth_valid":                  "Validity",
    "country_code_valid":                   "Validity",
    "customer_id_uuid_format":              "Validity",
    "created_at_not_future":                "Consistency",
    "date_of_birth_not_future":             "Consistency",
    "age_between_18_and_120":               "Consistency",
    "customer_id_unique":                   "Uniqueness",
 
    # Accounts
    "account_id_not_null":                  "Completeness",
    "customer_id_not_null":                 "Completeness",
    "account_type_not_null":                "Completeness",
    "status_not_null":                      "Completeness",
    "account_type_valid":                   "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "account_id_uuid_format":               "Validity",
    "opened_date_valid":                    "Validity",
    "savings_loan_balance_not_negative":    "Consistency",
    "opened_date_not_future":               "Consistency",
    "account_id_unique":                    "Uniqueness",
    "customer_id_exists_in_customers":      "Referential Integrity",
 
    # Transactions
    "transaction_id_not_null":              "Completeness",
    "account_id_not_null":                  "Completeness",
    "transaction_type_not_null":            "Completeness",
    "amount_not_null":                      "Completeness",
    "currency_not_null":                    "Completeness",
    "transaction_date_not_null":            "Completeness",
    "status_not_null":                      "Completeness",
    "transaction_type_valid":               "Validity",
    "status_valid":                         "Validity",
    "currency_valid":                       "Validity",
    "transaction_id_uuid_format":           "Validity",
    "amount_greater_than_zero":             "Validity",
    "transaction_date_not_future":          "Consistency",
    "failed_reversed_amount_not_positive":  "Consistency",
    "transaction_id_unique":                "Uniqueness",
    "account_id_exists_in_accounts":        "Referential Integrity",
}
 
print(f"Rule-dimension map loaded: {len(RULE_DIMENSION_MAP)} rules")


In [0]:
#Assigning Severity
def get_severity(failed_rules_str, rule_dimension_map, severity_map):
    """
    Given a comma-separated string of failed rule names,
    return the highest severity among all failed rules.
    """
    if not failed_rules_str:
        return "LOW"
 
    severity_rank = {"HIGH": 3, "MEDIUM": 2, "LOW": 1}
    max_severity  = "LOW"
    max_rank      = 1
 
    for rule in failed_rules_str.split(", "):
        rule = rule.strip()
        dimension = rule_dimension_map.get(rule, "Completeness")
        severity  = severity_map.get(dimension, "LOW")
        rank      = severity_rank.get(severity, 1)
        if rank > max_rank:
            max_rank     = rank
            max_severity = severity
 
    return max_severity
 
severity_udf = F.udf(
    lambda failed_rules: get_severity(failed_rules, RULE_DIMENSION_MAP, SEVERITY_MAP),
    T.StringType()
)
 
print("severity_udf registered.")

In [0]:
#Extracting Exceptions from a Silver Table
def extract_exceptions(silver_df, table_name: str, pk_col: str):
    """
    Extract failed records from a Silver table and format as exception log rows.
 
    Args:
        silver_df  : Silver Delta DataFrame
        table_name : name of the source table (customers/accounts/transactions)
        pk_col     : primary key column name for this table
    """
    failed_df = silver_df.filter(F.col("dq_status") == "FAIL")
 
    failed_df = failed_df \
        .withColumn("exception_id",  F.expr("uuid()")) \
        .withColumn("table_name",    F.lit(table_name)) \
        .withColumn("record_id",     F.col(pk_col).cast(T.StringType())) \
        .withColumn("severity",      severity_udf(F.col("failed_rules"))) \
        .withColumn("reported_at",   F.lit(REPORTED_AT))
 
    # Derive failed_dimensions: map each rule in failed_rules to its dimension
    # and collect unique dimensions as a comma-separated string
    def get_dimensions(failed_rules_str):
        if not failed_rules_str:
            return ""
        dims = set()
        for rule in failed_rules_str.split(", "):
            rule = rule.strip()
            dim  = RULE_DIMENSION_MAP.get(rule, "Unknown")
            dims.add(dim)
        return ", ".join(sorted(dims))
 
    dimensions_udf = F.udf(get_dimensions, T.StringType())
 
    failed_df = failed_df.withColumn(
        "failed_dimensions",
        dimensions_udf(F.col("failed_rules"))
    )
 
    # Select only the columns needed for the exception log
    return failed_df.select(
        "exception_id",
        "table_name",
        "record_id",
        "failed_rules",
        "failed_dimensions",
        "severity",
        "validated_at",
        "reported_at",
    )
 
print("extract_exceptions() helper defined.")

In [0]:
#Loading Silver Tables and Extract Exceptions
silver_customers    = spark.read.format("delta").table("silver_customers")
silver_accounts     = spark.read.format("delta").table("silver_accounts")
silver_transactions = spark.read.format("delta").table("silver_transactions")
 
# Extract exceptions from each table using its primary key
customer_exceptions    = extract_exceptions(silver_customers,    "customers",    "customer_id")
account_exceptions     = extract_exceptions(silver_accounts,     "accounts",     "account_id")
transaction_exceptions = extract_exceptions(silver_transactions, "transactions", "transaction_id")
 
print("Exceptions extracted:")
print(f"  customers    : {customer_exceptions.count():,} failed records")
print(f"  accounts     : {account_exceptions.count():,} failed records")
print(f"  transactions : {transaction_exceptions.count():,} failed records")

In [0]:
#Union and writing gold_exception_log
gold_exception_log = customer_exceptions \
    .union(account_exceptions) \
    .union(transaction_exceptions)
 
gold_exception_log.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_exception_log")
 
total_exceptions = gold_exception_log.count()
print(f"gold_exception_log written: {total_exceptions:,} total exceptions")

In [0]:
#Exception Summary
print("=" * 60)
print("EXCEPTION REPORT — SUMMARY")
print("=" * 60)
 
# By table and severity
print("\nExceptions by Table and Severity:")
gold_exception_log \
    .groupBy("table_name", "severity") \
    .agg(F.count("*").alias("exception_count")) \
    .orderBy("table_name", "severity") \
    .show(30, truncate=50)
 
# By dimension
print("Exceptions by Failed Dimension:")
gold_exception_log \
    .groupBy("table_name", "failed_dimensions") \
    .agg(F.count("*").alias("exception_count")) \
    .orderBy("table_name", F.desc("exception_count")) \
    .show(30, truncate=60)
 
# Top 10 most common failed rules across all tables
print("Top 10 Most Common Failed Rules:")
gold_exception_log \
    .groupBy("failed_rules") \
    .agg(F.count("*").alias("exception_count")) \
    .orderBy(F.desc("exception_count")) \
    .limit(10) \
    .show(truncate=80)

In [0]:
#Sample Exception Records
for severity in ["HIGH", "MEDIUM", "LOW"]:
    print(f"\nSample {severity} severity exceptions:")
    gold_exception_log \
        .filter(F.col("severity") == severity) \
        .select("table_name", "record_id", "failed_rules", "failed_dimensions", "severity") \
        .limit(3) \
        .show(truncate=60)
 
print("\nNotebook 05 complete.")
print("All 5 notebooks done. DataSentry pipeline is ready for Power BI.")